# Optional CSV-ingestion benchmark: pandas vs. RAPIDS cuDF

This notebook is only for benchmarking CSV ingestion. It does not define model features or train the classifier. The modeling pipeline uses `src/features.py` so CPU and GPU workflows share feature formulas.

**Do not install cuDF into an already configured runtime without checking CUDA compatibility.** Earlier runs encountered pandas/cuDF/CuPy version conflicts. If cuDF is unavailable, skip this optional benchmark and use the previously recorded result only as a historical single-run measurement.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import time
import pandas as pd

# Set this to the original 1M-row CSV if you have it. Parquet is not suitable for a CSV-ingestion benchmark.
CSV_PATH = Path('/content/drive/MyDrive/Colab Notebooks/student_mental_health_burnout_1M.csv')
if not CSV_PATH.exists():
    print('CSV_PATH does not exist. Set it to the original Kaggle CSV to run this benchmark.')
else:
    start = time.perf_counter()
    pandas_df = pd.read_csv(CSV_PATH)
    pandas_seconds = time.perf_counter() - start
    print(f'pandas: {pandas_seconds:.4f} s; shape={pandas_df.shape}')

    try:
        import cudf
    except Exception as exc:
        print('cuDF is not available in this runtime; skipping GPU read benchmark.')
        print('Reason:', repr(exc))
    else:
        start = time.perf_counter()
        cudf_df = cudf.read_csv(str(CSV_PATH))
        cudf_seconds = time.perf_counter() - start
        print(f'cuDF: {cudf_seconds:.4f} s; shape={cudf_df.shape}')
        print(f'Single-run ratio (pandas/cuDF): {pandas_seconds / cudf_seconds:.2f}x')
        print('Treat this as a single-run read benchmark; repeat under controlled conditions for a robust comparison.')


## Previously recorded result

A prior Google Colab T4 run recorded pandas at 4.9373 seconds and cuDF at 0.2279 seconds (about 21.67× for that single CSV read). These are historical timings and may not reproduce exactly on another runtime. They measure ingestion only, not feature engineering or model training.
